# CS383: Data Science and Machine Learning
## Lecture 5 Exercises — Exploratory Data Analysis and Foundational Statistics

**Make a copy of this notebook before you start** (do not edit this original). Fill in every `__________` blank, then run all cells top to bottom before submitting. This notebook is graded with Otter Grader — do not delete or modify the setup cells.

### Setup — NYC 311

Run this first — it rebuilds the NYC 311 dataset with `resolution_time_hours` computed, same as Lecture 5.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests

SOCRATA_URL_311 = "https://data.cityofnewyork.us/resource/erm2-nwe9.json"

try:
    response = requests.get(
        SOCRATA_URL_311,
        params={
            "$limit": 8000,
            "$order": "created_date DESC",
            "$select": "complaint_type,borough,created_date,closed_date",
        },
        timeout=8,
    )
    response.raise_for_status()
    complaints_df = pd.DataFrame(response.json())
    complaints_df["created_date"] = pd.to_datetime(complaints_df["created_date"])
    complaints_df["closed_date"] = pd.to_datetime(complaints_df["closed_date"])
    live = True

except Exception:
    # Offline fallback, in case there is no internet connection in the room.
    rng = np.random.default_rng(383)
    n = 8000
    complaint_types = ["Noise - Residential", "Illegal Parking", "HEAT/HOT WATER",
                        "Blocked Driveway", "Street Condition", "Water System",
                        "PAINT/PLASTER", "Damaged Tree", "Sewer", "Rodent"]
    boroughs_list = ["MANHATTAN", "BROOKLYN", "QUEENS", "BRONX", "STATEN ISLAND"]

    n_days = 90
    days = pd.date_range("2026-01-01", periods=n_days, freq="D")
    day_weight = np.where(days.dayofweek >= 5, 0.6, 1.0)
    day_weight = day_weight / day_weight.sum()

    day_idx = rng.choice(n_days, size=n, p=day_weight)
    created = days[day_idx] + pd.to_timedelta(rng.integers(0, 24 * 60, size=n), unit="m")

    still_open = rng.random(n) < 0.15
    resolution_hours = rng.gamma(shape=2.0, scale=20.0, size=n)
    closed = created + pd.to_timedelta(resolution_hours, unit="h")

    complaints_df = pd.DataFrame({
        "complaint_type": rng.choice(
            complaint_types, size=n,
            p=[0.18, 0.15, 0.14, 0.10, 0.10, 0.09, 0.08, 0.06, 0.05, 0.05],
        ),
        "borough": rng.choice(boroughs_list, size=n, p=[0.22, 0.32, 0.26, 0.16, 0.04]),
        "created_date": created,
        "closed_date": np.where(still_open, pd.NaT, closed),
    })
    complaints_df["closed_date"] = pd.to_datetime(complaints_df["closed_date"])
    live = False

complaints_df["resolution_time_hours"] = (
    complaints_df["closed_date"] - complaints_df["created_date"]
).dt.total_seconds() / 3600

print(f"{'Live' if live else 'Offline fallback'} data: {len(complaints_df):,} 311 records")
complaints_df.head()

---

## Exercise 1 — EDA and a Non-Technical Finding

**Scenario:** you've been asked for one finding about 311 complaint resolution times, written for a general audience — the exact shape of Assignment 2. Use IQR-based outlier detection, one chart, and a plain-language summary.

### Step 1 — Compute the IQR bounds for `resolution_time_hours`

In [ ]:
res = complaints_df["resolution_time_hours"].dropna()

q1, q3 = res.quantile([__________, __________])
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + __________ * iqr

print(f"Bounds: [{lower_bound:.1f}, {upper_bound:.1f}] hours")

### Step 2 — Find how many complaints are unusually slow to resolve

In [ ]:
slow_complaints = complaints_df[complaints_df["resolution_time_hours"] __________ upper_bound]
print(f"{len(slow_complaints):,} complaints flagged as unusually slow")

### Step 3 — Make one chart

In [ ]:
sns.__________(data=complaints_df, x="resolution_time_hours", kde=True)
plt.title("Distribution of 311 complaint resolution time")
plt.show()

### Step 4 — Write the plain-language summary

In 2-3 sentences, explain your finding to someone who doesn't know what IQR, a histogram, or `groupby` are.

**Your summary:**



---

## Exercise 2 — Reflection (Exit Ticket)

Answer the following in your own words.

1. Why can the mean and median of the same column give noticeably different values?
2. What does the IQR rule (1.5×IQR beyond Q1/Q3) actually flag, and why doesn't a flagged point automatically mean "delete this row"?
3. Explain, in your own words, why the restaurant score/violation-count correlation isn't really a "discovery."
4. Give one example (from today or otherwise) of two things that might be correlated because of a shared confounder, not because one causes the other.
5. What question do you still have about EDA or statistics before Assignment 2?

**Your responses:**

1.  
2.  
3.  
4.  
5.  

## Optional Challenge

Apply the same descriptive-statistics-to-correlation workflow from Lecture 5 to the restaurant inspections dataset.

### Setup — NYC restaurant inspections

In [ ]:
SOCRATA_URL_REST = "https://data.cityofnewyork.us/resource/43nn-pn8j.json"

try:
    response = requests.get(
        SOCRATA_URL_REST,
        params={
            "$limit": 15000,
            "$order": "inspection_date DESC",
            "$select": "camis,boro,cuisine_description,inspection_date,score,grade,"
                        "violation_code,critical_flag",
        },
        timeout=10,
    )
    response.raise_for_status()
    raw = pd.DataFrame(response.json())
    raw["score"] = pd.to_numeric(raw["score"], errors="coerce")

    inspections_df = (
        raw.groupby(["boro", "cuisine_description", "score", "grade"], dropna=False)
        .agg(
            violation_count=("violation_code", "count"),
            critical_count=("critical_flag", lambda s: (s == "Critical").sum()),
        )
        .reset_index()
    )
    inspections_df = inspections_df.dropna(subset=["score"]).reset_index(drop=True)
    live = True

except Exception:
    # Offline fallback, in case there is no internet connection in the room.
    rng = np.random.default_rng(383)
    n = 1200
    boroughs_list = ["MANHATTAN", "BROOKLYN", "QUEENS", "BRONX", "STATEN ISLAND"]
    cuisines_clean = ["American", "Chinese", "Italian", "Mexican", "Pizza",
                       "Japanese", "Caribbean", "Bakery", "Coffee/Tea", "Chicken"]

    violation_count = rng.poisson(lam=3, size=n)
    critical_count = rng.binomial(violation_count, 0.4)
    noise = rng.normal(0, 3, size=n)
    score = np.clip(violation_count * 7 + critical_count * 5 + noise, 0, 140).round().astype(int)

    grade = np.where(score <= 13, "A", np.where(score <= 27, "B", "C")).astype(object)
    ungraded_idx = rng.choice(n, size=int(n * 0.1), replace=False)
    grade[ungraded_idx] = None

    inspections_df = pd.DataFrame({
        "boro": rng.choice(boroughs_list, size=n, p=[0.22, 0.32, 0.26, 0.16, 0.04]),
        "cuisine_description": rng.choice(cuisines_clean, size=n),
        "score": score,
        "grade": grade,
        "violation_count": violation_count,
        "critical_count": critical_count,
    })
    live = False

print(f"{'Live' if live else 'Offline fallback'} data: {len(inspections_df):,} inspections")
inspections_df.head()

### Step 1 — Compute the mean, median, and standard deviation of `score`

In [ ]:
# Your code here


### Step 2 — Find the IQR outlier bounds and count how many inspections are flagged

In [ ]:
# Your code here


### Step 3 — Scatter plot `violation_count` vs. `score`, then compute the correlation coefficient

In [ ]:
# Your code here


### Step 4 — Reflect

Is this correlation closer to a genuine discovery, or closer to definitional (a result of how `score` was computed)? Explain in 1-2 sentences.

**Your answer:**

